In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check if GPU is available
print(f"ROCm version: {torch.version.hip}")
print(f"CUDA version: {torch.version.cuda}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ROCm version: None
CUDA version: 12.8
Using device: cpu


# Components

In [3]:
MAX_SEQ_LEN: int = 2048


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""

    def __init__(self, d_model: int, eps: float = 1e-8):
        """
        Initialize the RMSNorm layer.

        Args:
            d_model (int): The dimension of the model.
            eps (float): A small value to avoid division by zero.
        """
        super(RMSNorm, self).__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute the RMSNorm of the input tensor."""
        mean_square = x.pow(2).mean(-1, keepdim=True)
        rms = torch.sqrt(mean_square + self.eps)
        x_normed = x / rms
        return x_normed * self.scale


class SwiGLU(nn.Module):
    """SwiGLU Activation Function."""

    def __init__(self, input_dim: int, output_dim: int):
        """
        Initialize the SwiGLU layer.

        Args:
            input_dim (int): The dimension of the input.
            output_dim (int): The dimension of the output.
        """
        super(SwiGLU, self).__init__()
        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute the SwiGLU activation."""
        gates = self.fc1(x)
        values = self.fc2(x)
        return gates * F.silu(values)


class RoPE(nn.Module):
    """Rotary Position Embedding."""

    def __init__(self, d_model: int, max_seq_len: int = MAX_SEQ_LEN):
        """
        Initialize the RoPE layer.

        Args:
            d_model (int): The dimension of the model.
            max_seq_len (int): Maximum sequence length.
        """
        super(RoPE, self).__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Precompute frequencies
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        """Apply rotary position embeddings."""
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)

        cos_emb = emb.cos()[None, None, :, :]
        sin_emb = emb.sin()[None, None, :, :]

        x_rot = torch.stack([-x[..., 1::2], x[..., ::2]], dim=-1).flatten(-2)
        return x * cos_emb + x_rot * sin_emb


class SelfAttention(nn.Module):
    """Multi-Head Self Attention with RoPE."""

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int = MAX_SEQ_LEN):
        """
        Initialize the Self Attention layer.

        Args:
            d_model (int): The dimension of the model.
            num_heads (int): The number of attention heads.
            max_seq_len (int): Maximum sequence length.
        """
        super(SelfAttention, self).__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        # Calculate dimension per attention head
        self.head_dim = d_model // num_heads

        # Linear projections for query, key, value, and output
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        # Initialize RoPE for positional embeddings
        self.rope = RoPE(self.head_dim, max_seq_len)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """Compute multi-head self attention."""
        # Get batch size and sequence length
        batch_size, seq_len, _ = x.shape

        # Project input and reshape for multi-head attention
        q = (
            self.q_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        k = (
            self.k_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        v = (
            self.v_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )

        # Apply rotary position embeddings to queries and keys
        q = self.rope(q, seq_len)
        k = self.rope(k, seq_len)

        # Compute attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim**0.5)

        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # Compute attention weights
        attn_weights = F.softmax(scores, dim=-1)
        # Apply attention weights to values
        context = torch.matmul(attn_weights, v)

        # Reshape context back to original dimensions
        context = (
            context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        )
        # Project output
        output = self.out_proj(context)

        return output